# Ask an agent to explore AOPWiki

Can a model use our generated classes to answer a research question? We give it the same tools we use: list classes, find names, and follow links. We then compare its proposed query with an independently written query.

Start with the saved session from `pydantic_clients/01_mine_explore.ipynb`. We reuse its classes with a fixed local AOPWiki dump. No requests go to the public endpoint.

## Set up once

In the project environment, run `uv pip install -e '.[notebooks,agents]'` and select the **rdfsolve (Python 3.12)** kernel.

Copy `.env.example` to `.env` in this folder and add your API keys. `.env` and experiment outputs are ignored by Git. Keep keys out of notebook cells.

In [1]:
from pathlib import Path
from rdfsolve.client_api import Client
from rdfsolve.pydantic_ai import ClientTools

session = Path('../pydantic_clients/thyroid-session.json')
dump = Path('../../../data/qlever_workdirs/aopwikirdf/AOPWikiRDF.ttl')

data = Client.from_session(session, data_file=dump)
data.types()

,Class
0,Activity
1,Adverse Outcome Pathway
2,Biological event
3,CAS registry number
4,Cell-term
5,Ch EBI identifier
6,Ch EMBL identifier
7,Chem Spider identifier
8,Drug Bank identifier
9,Ensembl gene ID


## Give the model the same tools

`ClientTools(data)` exposes the classes and fields we generated. It can find records and follow their links. It keeps the actual typed records in Python and shows the model their names, classes, and identifiers.

In [2]:
tools = ClientTools(data)
tools.find('Phenobarbital', kind='Chemical entity')

{'reference': 'r1',
 'rows': [{'Name': 'Phenobarbital',
   'Class': 'chemical entity',
   'IRI': 'https://identifiers.org/cas/50-06-6'}],
 'total_rows': 1,
 'preview_only': False}

An agent can use this toolset in a few lines:

```python
from pydantic_ai.usage import UsageLimits

agent = tools.agent('openai:gpt-5.4-mini-2026-03-17')
answer = await agent.run(
    'Which classes describe Phenobarbital, and what links can I follow from it?',
    usage_limits=UsageLimits(request_limit=8, tool_calls_limit=12),
)
print(answer.output)
```

The comparison below runs the agents for us and saves their queries and results.

## Compare answers

We start with four questions from AOP-Wiki-Queries: count pathways, find pathways for an adverse outcome, find chemical identifiers, and find chemicals for a pathway.

Models receive only the question, parameter values, and requested columns. They do not receive reference queries or answers. Both sets of queries run against a frozen copy of the dump. This base dump lacks some enrichment: an empty reference answer is reported as empty, not as evidence of good recall.

In [3]:
from experiment import prepare, configured_models, compare

run = prepare(dump, session)
models = configured_models()
print('Models ready:', models or 'Add your keys to .env to run the comparison.')
print('Saved experiment:', run)

Models ready: Add your keys to .env to run the comparison.
Saved experiment: /trinity/home/javier.millanacosta/rdfsolve/rdfsolve-2/notebooks/pydantic_ai/runs/20260908-155342


In [4]:
scores = await compare(run, models)
scores

,model,case,expected_rows,status,exact,precision,recall,f1,seconds


`exact` means the returned rows match, including repeated rows. Precision asks how many returned rows belong in the answer; recall asks how much of the reference answer was recovered. Counts use exact equality instead.

Failed attempts stay in the table. Four questions and one attempt per model are a starting point, not a model ranking. See the folder README for the selected questions, exclusions, and run limits.

## See what was actually queried

Our own exploration is recorded below. The comparison also saves each agent's conversation, proposed query, returned rows, usage, and client query log in `results.json` inside the experiment folder.

In [5]:
data.query_log()

QueryLog(2 executions; open in a notebook to view results)

In [6]:
data.close()